# 00 · Pipeline overview & configuration

This is the entry point for the **MILK** (Multi-Instance Learning Kit) notebook family.
Each notebook launches one step of the pipeline and can be run on its own:

| Notebook | Step | What it does |
|----------|------|--------------|
| `00_overview_and_config.ipynb` | Setup | Loads & inspects the YAML config (this notebook) |
| `01_data_preprocessing.ipynb` | Data preprocessing | Builds bags, clusters conformers, scales features via `MILDataModule.setup()` |
| `02_model_construction.ipynb` | Model construction | Builds the MIL model (embedder → aggregator → predictor) |
| `03_model_training.ipynb` | Training & evaluation | Trains on train, validates on val, tests on test (predefined split) |

The data uses a **predefined split** (`split` column: 0=train, 1=val, 2=test) —
no cross-validation, no stages. All four notebooks read the **same** `CONFIG_PATH`, so
changing the config in one place keeps every step consistent. Under the hood
these notebooks call the exact same classes as `poetry run milk -c <config>` —
nothing is re-implemented.

In [1]:
# --- Bootstrap: make the notebook run from anywhere ---
import os, sys, logging
from pathlib import Path

# Locate the project root (folder that contains the `ppl` package).
here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [here, *here.parents] if (p / 'ppl' / '__init__.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    # Fallback: this notebook lives in <root>/notebooks/
    PROJECT_ROOT = Path('__file__' in globals() and __file__ or '.').resolve().parent.parent

os.chdir(PROJECT_ROOT)                       # pipeline writes outputs relative to cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
print('Project root:', PROJECT_ROOT)

Project root: /Users/vfastovskii/Desktop/milkid


In [2]:
# Path to the experiment YAML. Edit this to point at a different config.
CONFIG_PATH = PROJECT_ROOT / 'ppl/config/experiment_configs/run_config.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH.relative_to(PROJECT_ROOT))

Using config: ppl/config/experiment_configs/run_config.yaml


## Load the config

`PipelineConfig.from_yaml` validates the YAML and builds the three config
sections (`data`, `model`, `trainer`). `PipelineConfigManager` then applies
defaults/overrides exactly as the real pipeline does.

In [3]:
from ppl.config.pipeline_config import PipelineConfig
from ppl.pipeline.config_manager import PipelineConfigManager

cfg = PipelineConfig.from_yaml(CONFIG_PATH)
cfg_mgr = PipelineConfigManager(cfg)

print('task      :', cfg_mgr.task)
print('seed      :', cfg_mgr.seed)
print('log dir   :', cfg_mgr.log_save_dir)
print('experiment:', cfg_mgr.trainer_cfg.experiment_name)

INFO ppl.config.pipeline_config: Using default csv_path: /Users/vfastovskii/Desktop/milkid/ppl/data/trypsin_usrcat.csv
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to DataLoaderConfig
INFO ppl.pipeline.config_manager: Using experiment_name 'bace809_cluster_hier_mha_200c_2jul_bs16_test15' for data splits
INFO ppl.pipeline.config_manager: Setting default cache_dir to 'bace809_cluster_hier_mha_200c_2jul_bs16_test15' for data splits
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to ModelBuilderConfig
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to TrainerOptimConfig
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to TrainerConfig
INFO ppl.pipeline.config_manager: Setting global seed to 42
INFO ppl.utils.reproducibility.deterministic_setup: Deterministic setup complete with seed: 42
INFO ppl.utils.reproducibility.deterministic_setup: CUBLAS_WORKSPACE_CONFIG = :4096:8
INFO ppl.utils.reproducibility.det

task      : regression
seed      : 42
log dir   : bace809_cluster_hier_mha_200c_2jul_bs16_test15_log
experiment: bace809_cluster_hier_mha_200c_2jul_bs16_test15


### Data configuration

In [4]:
import dataclasses as dc, pprint
pprint.pp(dc.asdict(cfg_mgr.data_cfg))

{'csv_path': PosixPath('ppl/data/bace809_3d_fp_5fp_200c_05prune_6kcal_no_Hs_june_concated_splited.csv'),
 'task': 'regression',
 'descriptor_cols': None,
 'bag_id_col': 'mol_id',
 'inst_id_col': 'conf_id',
 'endpoint_value_col': 'endpoint',
 'energy_col': 'Energy',
 'pdb_id_col': 'pdb_id',
 'smiles_col': 'smiles',
 'split_col': 'split',
 'series_col': 'Series',
 'predefined_split': True,
 'test_size': 0.2,
 'n_strat_bins': 5,
 'seed': 42,
 'batch_size': 16,
 'num_workers': 10,
 'pin_memory': True,
 'balance_train_batches_by_series': True,
 'cluster_instances': True,
 'cluster_selection_method': 'silhouette',
 'cluster_max_clusters': 6,
 'cluster_min_clusters': 1,
 'cluster_min_silhouette': 0.08,
 'cluster_distance_threshold': None,
 'cluster_linkage': 'average',
 'cluster_metric': 'euclidean',
 'cache_dir': PosixPath('bace809_cluster_hier_mha_200c_2jul_bs16_test15'),
 'memory_limit': None,
 'on_demand_loading': False,
 'experiment_name': 'bace809_cluster_hier_mha_200c_2jul_bs16_test15'

### Model configuration

In [5]:
print('embedder_type   :', cfg_mgr.model_cfg.embedder_type)
print('aggregator_type :', cfg_mgr.model_cfg.aggregator_type)
print('predictor_type  :', cfg_mgr.model_cfg.predictor_type)

embedder_type   : contextualized_mlp_embedder_v1
aggregator_type : cluster_hier_mha_v1
predictor_type  : mlp_predictor_v3


### Trainer configuration

In [6]:
print('max_epochs      :', cfg_mgr.trainer_cfg.max_epochs)
print('device          :', cfg_mgr.trainer_cfg.device)
print('precision       :', cfg_mgr.trainer_cfg.precision)
print('checkpoint_monitor:', cfg_mgr.trainer_cfg.checkpoint_monitor)

max_epochs      : 100
device          : mps
precision       : 32-true
checkpoint_monitor: val_rmse


---
Config looks good? Continue with **`01_data_preprocessing.ipynb`**.